# SETUP

In [1]:
#!pip3 uninstall --yes tensorflow jax jaxlib torch torch_xla torchvision torchaudio torchtext torchdata ml-dtypes
#!pip3 cache purge

#!pip3 install torch~=2.5.0 torch_xla~=2.5.0 https://storage.googleapis.com/pytorch-xla-releases/wheels/cuda/12.1/torch_xla_cuda_plugin-2.5.0-py3-none-any.whl torchvision torchaudio torchtext torchdata ml-dtypes
#!pip3 install -U "jax[cuda12]"

In [2]:
import jax
import jax.numpy as jnp
import numpy as np
import torch
import torch_xla.core.xla_model as xm
import time

In [3]:
print("Is CUDA available? ", torch.cuda.is_available())
print("CUDA version: ", torch.version.cuda)
print("PyTorch version: ", torch.__version__)
if torch.cuda.is_available():
    print("GPU device count:", torch.cuda.device_count())
    print("Current GPU device index:", torch.cuda.current_device())
    print("GPU device name:", torch.cuda.get_device_name(torch.cuda.current_device()))

print('\n')

print("JAX version:", jax.__version__)
print("JAX is using CUDA:", jax.lib.xla_bridge.get_backend().platform == 'gpu')
print("Available devices:", jax.devices())
x = jnp.ones((1000, 1000))
y = jnp.dot(x, x)
print("Computation device:", y.device)

<ipython-input-3-0f0eade4508b>:12: DeprecationWarning: jax.lib.xla_bridge.get_backend is deprecated; use jax.extend.backend.get_backend.
  print("JAX is using CUDA:", jax.lib.xla_bridge.get_backend().platform == 'gpu')


Is CUDA available?  True
CUDA version:  12.4
PyTorch version:  2.5.1+cu124
GPU device count: 1
Current GPU device index: 0
GPU device name: Tesla T4


JAX version: 0.6.0
JAX is using CUDA: True
Available devices: [CudaDevice(id=0)]
Computation device: cuda:0


# COMPARE JITs

In [4]:
# input tensors
A_np = np.random.rand(100, 100)
B_np = np.random.rand(100, 100)

A_jax = jnp.array(A_np)
B_jax = jnp.array(B_np)
A_torch = torch.from_numpy(A_np)
B_torch = torch.from_numpy(B_np)



# functions for native JIT
def jax_function(A, B):
	C = B
	for _ in range(50):
		C = jnp.matmul(A, C)
		C = (C + A + B) / 3
		C = C * A * B
		C = jnp.transpose(C)
		C = jnp.linalg.inv(C)
	d = jnp.linalg.det(C)
	return d

def pyt_function(A, B):
	C = B
	for _ in range(50):
		C = torch.matmul(A, C)
		C = (C + A + B) / 3
		C = C * A * B
		C = C.T
		C = torch.linalg.inv(C)
	d = torch.linalg.det(C)
	return d

# trace & warmup for native JIT
jax_function_jit = jax.jit(jax_function)
pyt_function_jit = torch.jit.script(pyt_function)
for _ in range(100): yt = jax_function_jit(A_jax, B_jax).block_until_ready()
for _ in range(100): yt = pyt_function_jit(A_torch, B_torch)

In [5]:
# functions for JIT+XLA
def jax_function_alt(A, B):
	C = B
	for _ in range(50):
		C = jnp.matmul(A, C)
		C = (C + A + B) / 3
		C = C * A * B
		C = jnp.transpose(C)
	return jnp.sum(C)

def pyt_function_alt(A, B):
	C = B
	for _ in range(50):
		C = torch.matmul(A, C)
		C = (C + A + B) / 3
		C = C * A * B
		C = C.T
	return torch.sum(C)

# trace & warm up for JIT+XLA
device = xm.xla_device()
A_torch_xla = A_torch.to(device)
B_torch_xla = B_torch.to(device)

jax_function_alt_jit = jax.jit(jax_function_alt)
pyt_function_alt_jit = torch.jit.script(pyt_function_alt)
for _ in range(100): yt = jax_function_alt_jit(A_jax, B_jax).block_until_ready()
for _ in range(100): yt = pyt_function_alt_jit(A_torch_xla, B_torch_xla)

In [11]:
# no JIT

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function(A_jax, B_jax)
	end = time.time()
	runtimes.append(end - start)
print(f"JAX avg time: {sum(runtimes)/len(runtimes):.6f} seconds")

runtimes = list()
for _ in range(100):
	start = time.time()
	y = pyt_function(A_torch, B_torch)
	end = time.time()
	runtimes.append(end - start)
print(f"PyT avg time: {sum(runtimes)/len(runtimes):.6f} seconds")

JAX avg time: 0.061440 seconds
PyT avg time: 0.023023 seconds


In [12]:
# native JIT (XLA for JAX, TorchScript for PyTorch)

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function_jit(A_jax, B_jax).block_until_ready()
	end = time.time()
	runtimes.append(end - start)
print(f"JAX avg time (XLA): {sum(runtimes)/len(runtimes):.6f} seconds")

runtimes = list()
for _ in range(100):
	start = time.time()
	y = pyt_function_jit(A_torch, B_torch)
	end = time.time()
	runtimes.append(end - start)
print(f"PyT avg time  (TS): {sum(runtimes)/len(runtimes):.6f} seconds")

JAX avg time (XLA): 0.013163 seconds
PyT avg time  (TS): 0.023264 seconds


In [13]:
# JIT + XLA

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function_alt_jit(A_jax, B_jax).block_until_ready()
	end = time.time()
	runtimes.append(end - start)
print(f"Jax avg time (XLA): {sum(runtimes)/len(runtimes):.6f} seconds")

runtimes = list()
for _ in range(100):
	start = time.time()
	y = pyt_function_alt_jit(A_torch_xla, B_torch_xla)
	end = time.time()
	runtimes.append(end - start)
print(f"PyT avg time (XLA): {sum(runtimes)/len(runtimes):.6f} seconds")

Jax avg time (XLA): 0.001259 seconds
PyT avg time (XLA): 0.003142 seconds


# COMPARE AUTOGRADS

In [14]:
# grad, trace & warmup for native JIT
jax_function_grad = jax.grad(jax_function)
A_torch_grad = A_torch.requires_grad_(True)
B_torch_grad = B_torch.requires_grad_(True)

jax_function_grad_jit = jax.jit(jax_function_grad)
for _ in range(100): yt = jax_function_grad_jit(A_jax, B_jax).block_until_ready()
for _ in range(100): yt = pyt_function_jit(A_torch_grad, B_torch_grad).backward()

In [15]:
# grad, trace & warmup for JIT+XLA
jax_function_alt_grad = jax.grad(jax_function_alt)

device = xm.xla_device()
A_torch_grad_xla = A_torch_grad.to(device)
B_torch_grad_xla = B_torch_grad.to(device)

jax_function_alt_grad_jit = jax.jit(jax_function_alt_grad)
for _ in range(100): yt = jax_function_alt_grad_jit(A_jax, B_jax).block_until_ready()
for _ in range(100): yt = pyt_function_alt_jit(B_torch_grad_xla, B_torch_grad_xla).backward()

RuntimeError: Bad StatusOr access: FAILED_PRECONDITION: DNN library initialization failed. Look at the errors above for more details.

In [16]:
# Autograd native JIT

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function_grad_jit(A_jax, B_jax)
	end = time.time()
	runtimes.append(end - start)
print(f"Jax avg time (XLA): {sum(runtimes)/len(runtimes):.6f} seconds")

runtimes = list()
for _ in range(100):
	start = time.time()
	y = pyt_function_jit(A_torch_grad, B_torch_grad).backward()
	end = time.time()
	runtimes.append(end - start)
print(f"PyT avg time  (TS): {sum(runtimes)/len(runtimes):.6f} seconds")

Jax avg time (XLA): 0.018128 seconds
PyT avg time  (TS): 0.050221 seconds


In [17]:
# Autograd JIT+XLA

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function_alt_grad_jit(A_jax, B_jax)
	end = time.time()
	runtimes.append(end - start)
print(f"Jax avg time (XLA): {sum(runtimes)/len(runtimes):.6f} seconds")

runtimes = list()
for _ in range(100):
	start = time.time()
	y = pyt_function_alt_jit(A_torch_grad_xla, B_torch_grad_xla).backward()
	end = time.time()
	runtimes.append(end - start)
print(f"PyT avg time  (TS): {sum(runtimes)/len(runtimes):.6f} seconds")


Jax avg time (XLA): 0.001820 seconds


RuntimeError: Bad StatusOr access: FAILED_PRECONDITION: DNN library initialization failed. Look at the errors above for more details.